In [1]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.2 s3fs==2026.3.0 -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Imports

In [2]:
import pandas as pd
import io
import openpyxl
from openpyxl.drawing.image import Image

# Chargement des données

In [3]:

df_effectifs = pd.read_csv('../DATA/effectifs.csv', sep=';')
df_depenses = pd.read_csv('../DATA/depenses.csv', sep=';')

df_fusion = df_effectifs.merge(df_depenses,
                               on=['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top'], 
                               how='inner')


df_fusion.head()

,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop_x,...,tri_x,dep_niv_1,dep_niv_2,montant,Ntop_y,N_recourant_au_poste,montant_moy,Niveau prioritaire_y,tri_y,type_somme
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,...,78.0,Hospitalisations (tous secteurs),Hospitalisations en HAD secteur privé remboursées,406099,9650.0,70.0,42.0,3,78.0,Partiel
1,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,...,78.0,Hospitalisations (tous secteurs),Hospitalisations en psychiatrie secteur public...,0,9650.0,50.0,0.0,3,78.0,Partiel
2,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,...,78.0,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,69103,9650.0,60.0,7.0,3,78.0,Partiel
3,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,...,78.0,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur publ...,412221,9650.0,380.0,43.0,3,78.0,Partiel
4,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,...,78.0,Hospitalisations (tous secteurs),Hospitalisations séjour MCO secteur public rem...,20486382,9650.0,8080.0,2122.0,3,78.0,Partiel


In [5]:
df_fusion.dtypes

annee                     int64
patho_niv1                  str
patho_niv2                  str
patho_niv3                  str
top                         str
cla_age_5                   str
sexe                      int64
region                    int64
dept                        str
Ntop_x                  float64
Npop                      int64
prev                    float64
Niveau prioritaire_x        str
libelle_classe_age          str
libelle_sexe                str
tri_x                   float64
dep_niv_1                   str
dep_niv_2                   str
montant                   int64
Ntop_y                  float64
N_recourant_au_poste    float64
montant_moy             float64
Niveau prioritaire_y        str
tri_y                   float64
type_somme                  str
dtype: object

# Nettoyage des données

In [9]:
# Suppression des colonnes non utilisés
df = df_fusion.drop(columns=['libelle_classe_age','libelle_sexe','tri_x','tri_y','dep_niv_2', 'Niveau prioritaire_x','Niveau prioritaire_y','type_somme','libelle_classe_age'])


In [8]:
df

,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop_x,Npop,prev,dep_niv_1,montant,Ntop_y,N_recourant_au_poste,montant_moy
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,15550,NaN,Hospitalisations (tous secteurs),406099,9650.0,70.0,42.0
1,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,15550,NaN,Hospitalisations (tous secteurs),0,9650.0,50.0,0.0
2,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,15550,NaN,Hospitalisations (tous secteurs),69103,9650.0,60.0,7.0
3,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,15550,NaN,Hospitalisations (tous secteurs),412221,9650.0,380.0,43.0
4,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,28,999,NaN,15550,NaN,Hospitalisations (tous secteurs),20486382,9650.0,8080.0,2122.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161708395,2015,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,1,99,999,26800.0,2091600,1.281,Prestations en espèces,5366439,447980.0,6840.0,12.0
161708396,2015,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,1,99,999,26800.0,2091600,1.281,Prestations en espèces,279883142,447980.0,83290.0,625.0
161708397,2015,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,1,99,999,26800.0,2091600,1.281,Soins de ville,0,447980.0,2280.0,0.0
161708398,2015,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,1,99,999,26800.0,2091600,1.281,Soins de ville,11797910,447980.0,163400.0,26.0


In [6]:
df = df[
    (df["sexe"].astype(str) != "9") &
    (df["dept"].astype(str) != "999") & #On passe les departements en texte car j'ai des valeurs pour la corse en strinf "2A" etcs
    (~df["patho_niv3"].astype(str).str.contains(
        "Total|Pas de pathologie repérée|traitement|maternité", #ces termes sont trop long et rende l'affichage trop petit donc autant les enlevés
        case=False,
        na=False
    ))
]


# DATA VIZ - EXCEL

In [10]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.utils.dataframe import dataframe_to_rows

df = pd.read_csv("../DATA/effectifs.csv", sep=";")
df.columns = df.columns.str.strip()

df["libelle_sexe"] = df["libelle_sexe"].astype(str).str.strip().str.lower()
df["patho_niv3"] = df["patho_niv3"].astype(str).str.strip()

df["patho_niv3"] = df["patho_niv3"].str.replace(r"\(.*?\)", "", regex=True)
df["patho_niv3"] = df["patho_niv3"].str.replace(
    r"^Autres\s+.*", "Autres pathologies", regex=True, case=False
)
df["patho_niv3"] = df["patho_niv3"].str.replace(r"\s+", " ", regex=True).str.strip()

df_filtre = df[df["libelle_sexe"].isin(["hommes", "femmes"])]
df_filtre = df_filtre[
    ~df_filtre["patho_niv3"].str.contains(
        "Total|Pas de pathologie", na=False, case=False
    )
]

df_pivot = (
    df_filtre.pivot_table(
        index="patho_niv3", columns="libelle_sexe", values="Ntop", aggfunc="sum"
    )
    .fillna(0)
)

df_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

wb = Workbook()
ws = wb.active
ws.title = "Data_Pathologies"
ws.views.sheetView[0].showGridLines = True

labels_dynamiques = [str(col).capitalize() for col in df_pct.columns]
df_excel = df_pct.reset_index()
df_excel.columns = ["Pathologies"] + [f"{l} (%)" for l in labels_dynamiques]

for r in dataframe_to_rows(df_excel, index=False, header=True):
    ws.append(r)

max_row = len(df_excel) + 1

chart = BarChart()
chart.type = "bar"
chart.grouping = "stacked"
chart.overlap = 100
chart.title = "Répartition en % par sexe des pathologies"
chart.style = 10

data = Reference(ws, min_col=2, min_row=1, max_col=3, max_row=max_row)
cats = Reference(ws, min_col=1, min_row=2, max_row=max_row)

chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
chart.y_axis.scaling.orientation = "maxMin"

chart.width = 18
chart.height = 12
ws.add_chart(chart, "E2")

for col in ws.iter_cols(max_col=3):
    max_len = max(len(str(cell.value or "")) for cell in col)
    col_letter = col[0].column_letter
    ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

wb.save("Analyses_Pathologies.xlsx")